In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()
df = df.drop(columns=["customerID"])
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df["Churn"].value_counts()

Churn
0    5163
1    1869
Name: count, dtype: int64

In [3]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

print(X.shape)
print(y.shape)

(7032, 19)
(7032,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

(5625, 19)
(1407, 19)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

colunas_texto = X_train.select_dtypes(include="object").columns
colunas_numero = X_train.select_dtypes(exclude="object").columns

preparo = ColumnTransformer([
    ("texto", OneHotEncoder(handle_unknown="ignore"), colunas_texto),
    ("numero", StandardScaler(), colunas_numero),
])

modelo = Pipeline([
    ("preparo", preparo),
    ("modelo", LogisticRegression(max_iter=1000)),
])

modelo.fit(X_train, y_train)
print("Acerto na prova:", modelo.score(X_test, y_test))

Acerto na prova: 0.8038379530916845


/tmp/ipykernel_20521/2368117373.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colunas_texto = X_train.select_dtypes(include="object").columns


In [6]:
from sklearn.metrics import classification_report

previsoes = modelo.predict(X_test)
print(classification_report(y_test, previsoes))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



In [7]:
import os
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

os.makedirs("images", exist_ok=True)

# Imagem 1: quantos cancelaram
fig, ax = plt.subplots(figsize=(6, 4))
contagem = df["Churn"].map({0: "Permaneceu", 1: "Cancelou"}).value_counts()
ax.bar(contagem.index, contagem.values, color=["#4C72B0", "#C44E52"])
ax.set_title("Distribuição de clientes")
ax.set_ylabel("Número de clientes")
fig.tight_layout()
fig.savefig("images/distribuicao_churn.png", dpi=150)
plt.close(fig)

# Imagem 2: taxa de churn por tipo de contrato
fig, ax = plt.subplots(figsize=(7, 4))
taxa = df.groupby("Contract")["Churn"].mean().sort_values(ascending=False) * 100
ax.bar(taxa.index, taxa.values, color="#4C72B0")
ax.set_title("Taxa de churn por tipo de contrato")
ax.set_ylabel("Clientes que cancelaram (%)")
fig.tight_layout()
fig.savefig("images/churn_por_contrato.png", dpi=150)
plt.close(fig)

# Imagem 3: matriz de confusão
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_estimator(
    modelo, X_test, y_test,
    display_labels=["Permaneceu", "Cancelou"],
    cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title("Matriz de confusão (teste)")
fig.tight_layout()
fig.savefig("images/matriz_confusao.png", dpi=150)
plt.close(fig)

print("Imagens salvas!")

Imagens salvas!


In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

modelo_rf = Pipeline([
    ("preparo", preparo),
    ("modelo", RandomForestClassifier(n_estimators=200, random_state=42)),
])

modelo_rf.fit(X_train, y_train)
previsoes_rf = modelo_rf.predict(X_test)
print(classification_report(y_test, previsoes_rf))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.62      0.50      0.55       374

    accuracy                           0.79      1407
   macro avg       0.73      0.69      0.71      1407
weighted avg       0.78      0.79      0.78      1407



In [9]:
from sklearn.metrics import precision_score, recall_score

probs = modelo.predict_proba(X_test)[:, 1]

for limiar in [0.5, 0.4, 0.3, 0.2]:
    prev = (probs >= limiar).astype(int)
    p = precision_score(y_test, prev)
    r = recall_score(y_test, prev)
    print(f"limiar {limiar}: recall {r:.2f} | precision {p:.2f}")

limiar 0.5: recall 0.57 | precision 0.65
limiar 0.4: recall 0.68 | precision 0.58
limiar 0.3: recall 0.76 | precision 0.51
limiar 0.2: recall 0.87 | precision 0.46


In [10]:
from sklearn.metrics import classification_report

prev_03 = (probs >= 0.3).astype(int)
print(classification_report(y_test, prev_03))

              precision    recall  f1-score   support

           0       0.89      0.74      0.81      1033
           1       0.51      0.76      0.61       374

    accuracy                           0.74      1407
   macro avg       0.70      0.75      0.71      1407
weighted avg       0.79      0.74      0.76      1407



In [11]:
import pandas as pd

nomes = modelo.named_steps["preparo"].get_feature_names_out()
pesos = modelo.named_steps["modelo"].coef_[0]

importancia = pd.Series(pesos, index=nomes).sort_values()

print("Empurram PARA CANCELAR:")
print(importancia.tail(5))
print()
print("Empurram PARA FICAR:")
print(importancia.head(5))

Empurram PARA CANCELAR:
texto__PaymentMethod_Electronic check    0.180671
texto__StreamingTV_Yes                   0.191113
texto__InternetService_Fiber optic       0.590184
texto__Contract_Month-to-month           0.613846
numero__TotalCharges                     0.644014
dtype: float64

Empurram PARA FICAR:
numero__tenure               -1.352313
texto__Contract_Two year     -0.779030
texto__InternetService_DSL   -0.616132
numero__MonthlyCharges       -0.541006
texto__PaperlessBilling_No   -0.300387
dtype: float64


In [12]:
import joblib

joblib.dump(modelo, "modelo_churn.joblib")
print("Modelo salvo!")

Modelo salvo!
